In [ ]:
# %%capture
# !pip install -U corus==0.10.0 tiktoken==0.10.0 youtokentome==1.0.6 sentence_transformers==5.0.0 numpy==2.0.2 matplotlib==3.10.0 torch==2.6.0+cu124 transformers==4.55.0 datasets==4.0.0

## Full finetuning

In [ ]:
import os
import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from datasets import load_dataset

In [ ]:
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

In [ ]:
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model_name = "ai-forever/rugpt3medium_based_on_gpt2"
output_dir = "rugpt3-poetry-finetuned"

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("AnyaSchen/russian_poetry_with_keywords")
dataset

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'author', 'keywords'],
        num_rows: 7755
    })
})

In [ ]:
dataset["train"][5]

{'text': 'Прочёл.\nПошёл.\nМинуты с три –\nопять застрял\nу двух витрин.\nКакникак,\nа к школьным зданьям\nпришёл\nс огромным опозданьем.\nДверь на ключ.\nТолкнулся Влас –\nне пускают Власа\nв класс!\n',
 'author': 'Маяковский',
 'keywords': "['класс', 'дверь', 'прочесть', 'толкнуться', 'застрять']"}

In [ ]:
dataset.unique("author"), len(dataset['train'])

({'train': ['Маяковский', 'Тютчев', 'Блок', 'Eceнин', 'Пушкин']}, 7755)

In [ ]:
context_length = 128

outputs = tokenizer(
        dataset["train"][:2]['text'],
        truncation=True,
        max_length=context_length,
        return_overflowing_tokens=True,
        return_length=True,
)
outputs

{'overflowing_tokens': [[], []], 'num_truncated_tokens': [-66, -54], 'input_ids': [[677, 1145, 21984, 1864, 271, 2931, 510, 203, 353, 6094, 8339, 16, 203, 436, 2237, 4939, 811, 16, 203, 19051, 806, 13513, 28925, 18, 203, 6039, 282, 15959, 203, 42126, 18, 203, 443, 450, 717, 1004, 306, 1992, 16, 203, 44969, 34890, 392, 5, 203, 443, 2255, 3809, 505, 16, 203, 307, 4411, 203, 404, 3005, 203, 431, 2133, 7238, 18, 203], [1295, 23262, 30260, 309, 30770, 16, 203, 1212, 39753, 203, 2575, 288, 4920, 339, 203, 273, 1519, 16, 203, 275, 1808, 16, 203, 24333, 203, 5719, 485, 4272, 203, 32827, 331, 203, 1821, 414, 30760, 18, 203, 5798, 203, 280, 26212, 1073, 37832, 203, 413, 8293, 42306, 16, 9666, 460, 16, 203, 370, 1577, 203, 350, 1898, 975, 30, 203, 443, 21032, 5, 443, 203, 3001, 510, 203, 7774, 6833, 309, 9209, 18, 203]], 'length': [62, 74], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [ ]:
dataset['train'][0]

{'text': 'Влас Прогулкин —\nмилый мальчик,\nспать ложился,\nвзяв журнальчик.\nВсе в журнале\nинтересно.\n– Дочитаю весь,\nхоть тресну!\n– Ни отец его,\nни мать\nне могли\nзаставить спать.\n',
 'author': 'Маяковский',
 'keywords': "['спать', 'журнальчик', 'заставить', 'мальчик', 'мать']"}

In [ ]:
def tokenize(example):
    outputs = tokenizer(
        example['text'],
        truncation=True,
        max_length=context_length,
        return_overflowing_tokens=True,
        return_length=True,
    )

    input_batch = []
    for length, input_ids in zip(outputs["length"], outputs["input_ids"]):
        if length == context_length:
            input_batch.append(input_ids)
    return {"input_ids": input_batch}


tokenized_data = dataset.map(
    tokenize, batched=True, remove_columns=dataset["train"].column_names
)
tokenized_data

Map:   0%|          | 0/7755 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 1369
    })
})

In [ ]:
model = GPT2LMHeadModel.from_pretrained(model_name)
model.config.n_ctx = context_length
model_size = sum(t.numel() for t in model.parameters())
model_size

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.73G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.73G [00:00<?, ?B/s]

355871744

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [ ]:
out = data_collator([tokenized_data["train"][i] for i in range(5)])
for key in out:
    print(f"{key} shape: {out[key].shape}")

input_ids shape: torch.Size([5, 128])
attention_mask shape: torch.Size([5, 128])
labels shape: torch.Size([5, 128])


In [ ]:
def generation(model, tokenizer):
    model.eval()
    model = model.to(device)

    prompts = [
        "Весна, весна!",
        "Августовский вечер",
        "Море, волны и "
    ]

    print("\n" + "="*50)

    for prompt in prompts:
        inputs = tokenizer.encode(prompt, return_tensors='pt').to(device)

        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_length=256,
                num_return_sequences=1,
                temperature=0.8,
                do_sample=True,
                top_p=0.9,
                top_k=50,
                repetition_penalty=1.2,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        print(generated_text)
        print("="*50)

In [ ]:
generation(model, tokenizer)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Весна, весна! Пора любви и счастья!"  

Кругом цветы - тюльпаны, нарциссы... Но почему-то в сердце тоска. От тоски хочется петь, но никто не услышит меня… Я бы хотела услышать голос любимой песни под аккомпанемент этого прекрасного органа, который звучит по всей России каждый день с утра до вечера: «С днем рожденья тебя поздравляем!» И пусть у нас все будет хорошо! Мы ведь любим друг друга!!! В День рождения моей семьи мы желаем Вам всего самого наилучшего!!!!!! Будьте счастливы вместе!!!
И конечно же: Здоровья, Любви и Долголетия вам всем!!!!!!!!
Пусть сбудутся мечты о хорошем успехе (хотя бы на бумаге)....

Posted via LjBeetle
Комментарии скрыты.
Я только знаю что песня эта очень популярная и её знают миллионы людей по всему миру, а значит она тоже может стать частью вашего личного праздника!
Удачи!!!


27455249	nagibaigoro	2014-07-22 18:39:00	Что нужно знать об украинском майдане? Оригинал взят у  в Что необходимо знать об украинском майдане?Не так давно я писал пост
Августовский в

In [ ]:
args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    save_strategy="steps",
    save_steps=500,
    num_train_epochs=2,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    learning_rate=5e-4,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    data_collator=data_collator,
    train_dataset=tokenized_data["train"],
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-3103839852.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


TrainOutput(global_step=344, training_loss=2.9789056112599925, metrics={'train_runtime': 225.6044, 'train_samples_per_second': 12.136, 'train_steps_per_second': 1.525, 'total_flos': 635695625404416.0, 'train_loss': 2.9789056112599925, 'epoch': 2.0})

In [ ]:
generation(model, tokenizer)


Весна, весна!
Кружимся в тумане, свистя на метели.
На улице снежной – вечерний свет.
Скукаешь ли ты под окном своим?
Всплеск ледяных брызг и звонко бьет твой лед.
Ты идешь по улицам скользкими шагами;
Но как же странно: я вижу тебя...
И вот, проходя с твоей улыбкой нежной,
Я увидел слезы твои без ресниц...
Что ж теперь?!– Я подошел к окну твоему...
Льдина за льдиной звякнула о лед.
Знакомый голос меня прервал твое молчанье:
«Завтра... завтра... Я жду...»
И вдруг из твоих глаз моих струится слезинка.
И я, глядя в сумрак ночных улиц туманных,
Увидел вас вдвоем... Как будто две тени скользящие...
Так мы не любим друг друга, но я понял вас!..
Вы прошли... Вы – в сумрачней дня... И над вами вздыхает вьюга...
Моей любви нет... Но есть счастье... – и она в глазах ваших...
Мы идем... Мы идем до ночными шагами... Звезды гаснут... звезды гасли... Звезды погасших звезд...
Только
Августовский вечер,
          время звездных минут.
Твой поцелуй – и ночь без огня, —
и ты уходишь от меня...
И я с мо